# as-strided-windowing — worked example 3: Extract the main-diagonal band of a matrix with as_strided

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `as-strided-windowing`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`as_strided` is not limited to windows — any access pattern expressible as a base offset plus per-axis strides is reachable with zero copy. For a contiguous `(N, N)` matrix with `stride()==(N, 1)`, the main diagonal has stride `N+1` (one row down AND one column right per step). Reusing that idea you can carve a `(N-K+1, K)` view whose row `i` is the length-`K` diagonal run starting at `(i, i)`.

## Worked solution

**Goal.** From a contiguous `(N, N)` matrix `M`, build a `(N-K+1, K)` zero-copy view where row `i` is `[M[i,i], M[i+1,i+1], ..., M[i+K-1,i+K-1]]` — overlapping length-`K` segments sliding down the main diagonal.

1. **Read the strides.** `sR, sC = M.stride()`. For a contiguous matrix this is `(N, 1)`, but we read it so the code survives non-contiguous inputs.
2. **The diagonal step.** Moving from `M[r,c]` to `M[r+1,c+1]` advances `sR + sC` storage units. That combined stride is what walks *along* a diagonal.
3. **Output rows.** Each starting point `(i, i)` must leave room for `K` diagonal steps, so `i` ranges `0 .. N-K`, giving `N_out = N - K + 1` rows.
4. **Build the view.** `M.as_strided(size=(N_out, K), stride=(sR + sC, sR + sC))`. The outer axis moves the diagonal-segment start down by one full diagonal step; the inner axis walks the `K` elements of one segment — both use `sR + sC`.

**Why it works.** A diagonal is just another strided line through storage. Because the start-step and within-step are identical here (both are one diagonal hop), both strides equal `sR + sC`, mirroring the `(s, s)` pattern of an ordinary 1-D window but along the diagonal direction.

In [ ]:
def diagonal_strided_block(M: Tensor, K: int) -> Tensor:
    N = M.shape[0]
    sR, sC = M.stride()
    N_out = N - K + 1
    step = sR + sC
    return M.as_strided(size=(N_out, K), stride=(step, step))

t.manual_seed(0)
M = t.arange(16, dtype=t.float32).reshape(4, 4)
# diag is [0,5,10,15]; K=2 segments slide down it
blk = diagonal_strided_block(M, K=2)
print(blk)        # [[0,5],[5,10],[10,15]]
print(blk.shape)  # torch.Size([3, 2])